In [1]:
from langchain_google_genai import GoogleGenerativeAI
from getpass import getpass

api_key = getpass("Enter you Google API key: ")

Enter you Google API key:  ········


In [2]:
llm = GoogleGenerativeAI(model = "gemini-3.8-flash", google_api_key = api_key, temperature = 0.7)

In [3]:
from langchain_community.document_loaders import CSVLoader

csvloader = CSVLoader(file_path="codebasics_faqs.csv")

data = csvloader.load()

print(data[:2])

[Document(metadata={'source': 'codebasics_faqs.csv', 'row': 0}, page_content='prompt: I have never done programming in my life. Can I take this bootcamp?\nresponse: Yes, this is the perfect bootcamp for anyone who has never done coding and wants to build a career in the IT/Data Analytics industry or just wants to perform better in your current job or business using data.'), Document(metadata={'source': 'codebasics_faqs.csv', 'row': 1}, page_content='prompt: Why should I trust Codebasics?\nresponse: Till now 9000 + learners have benefitted from the quality of our courses. You can check the review section and also we have attached their LinkedIn profiles so that you can connect with them and ask directly.')]


C:\Users\acar\AppData\Local\Temp\ipykernel_26544\1579142481.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import CSVLoader


In [4]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

instructor_embedding = HuggingFaceEmbeddings()

vector_db = FAISS.from_documents(documents = data, embedding = instructor_embedding)

W1026 11:44:11.532000 26544 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/199 [00:01<?, ?it/s]

In [5]:
retriever = vector_db.as_retriever()

In [6]:
from langchain_core.prompts import PromptTemplate

prompt_template = """Given the following context and a question, generate an answer based on this context only. 
In the answer try to provide as much text as possible from "response" section in the source document context without making much changes. 
If the answer is not found in the context, kindly state "I don't know. "Don't try to make up an answer.

CONTEXT: {context}

QUESTION: {question}"""

prompt = PromptTemplate(
    template = prompt_template, input_variables={"context", "question"}
)

In [7]:
from langchain_classic.chains import RetrievalQA

chain = RetrievalQA.from_chain_type(llm = llm,
                                   chain_type = "stuff",
                                   retriever = retriever,
                                   input_key = "query",
                                   return_source_documents = True,
                                   chain_type_kwargs={"prompt":prompt})

In [8]:
chain("do you guys offer internship and is EMI available?")

C:\Users\acar\AppData\Local\Temp\ipykernel_26544\3593658026.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  chain("do you guys offer internship and is EMI available?")


{'query': 'do you guys offer internship and is EMI available?',
 'result': 'Based on the provided context:\n\n* For virtual internship: Yes\n* For EMI option: No',
 'source_documents': [Document(id='46df488d-534b-4054-957d-1948de703594', metadata={'source': 'codebasics_faqs.csv', 'row': 14}, page_content='prompt: Do you provide any virtual internship?\nresponse: Yes'),
  Document(id='7e6aac14-16da-40a9-a571-5b3a5eb51ee4', metadata={'source': 'codebasics_faqs.csv', 'row': 13}, page_content='prompt: Do we have an EMI option?\nresponse: No'),
  Document(id='a5aa3084-30ad-41e6-a993-0b62db3fe8f5', metadata={'source': 'codebasics_faqs.csv', 'row': 11}, page_content='prompt: Do you provide any job assistance?\nresponse: Yes, We help you with resume and interview preparation along with that we help you in building online credibility, and based on requirements we refer candidates to potential recruiters.'),
  Document(id='0fb83793-4c73-4ff6-888a-de241958a6cb', metadata={'source': 'codebasics_fa

In [9]:
chain("do you have any javasript course?")

GoogleRateLimitError: Error calling model 'models/gemini-3.8-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.\nPlease retry in 40m5.932422451s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description